# 🤖 RAG (Retrieval Augmented Generation) com HuggingFace

## 📌 Atividade Semanal #8 - Inteligência Artificial

---

### O que é RAG?

**RAG (Retrieval Augmented Generation)** é uma técnica que combina dois componentes:

1. **Retrieval (Recuperação):** busca informações relevantes em uma base de conhecimento (documentos, PDFs, sites, etc.)
2. **Generation (Geração):** usa um LLM (Large Language Model) para gerar uma resposta usando essas informações como contexto

### Por que usar RAG?

LLMs como GPT, Llama e Mistral têm duas limitações importantes:
- **Conhecimento limitado:** só sabem o que aprenderam até a data do treinamento
- **Alucinações:** podem inventar respostas quando não sabem algo

Com RAG, fornecemos ao LLM informações atualizadas e específicas no momento da pergunta, reduzindo alucinações e permitindo respostas sobre dados privados ou recentes.

### Arquitetura do nosso RAG

```
    Pergunta do usuário
          ↓
  [1] Embedding da pergunta (sentence-transformers)
          ↓
  [2] Busca por similaridade no FAISS (vector database)
          ↓
  [3] Contexto recuperado + Pergunta → Prompt
          ↓
  [4] LLM gera resposta (FLAN-T5)
          ↓
    Resposta final
```

### Stack utilizada

| Componente | Biblioteca | Modelo |
|---|---|---|
| Embeddings | `sentence-transformers` | `all-MiniLM-L6-v2` |
| Vector DB | `FAISS` (Facebook AI) | - |
| LLM | `transformers` (HuggingFace) | `google/flan-t5-base` |
| Orquestração | `langchain` | - |

## 1️⃣ Instalação das dependências

Aqui instalamos todas as bibliotecas que vamos usar. No Google Colab, basta rodar essa célula uma vez.

In [ ]:
# Instalação das bibliotecas necessárias
# -q significa 'quiet' (instala sem mostrar logs longos)
!pip install -q langchain langchain-community langchain-huggingface
!pip install -q faiss-cpu sentence-transformers transformers

## 2️⃣ Importação das bibliotecas

Carregamos as classes que vamos usar de cada biblioteca.

In [ ]:
# LangChain: framework que nos ajuda a conectar os componentes do RAG
from langchain_huggingface import HuggingFaceEmbeddings, HuggingFacePipeline
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate

# Transformers: biblioteca da HuggingFace para carregar modelos de IA
from transformers import pipeline

print('✅ Bibliotecas importadas com sucesso!')

## 3️⃣ Base de Conhecimento

Aqui definimos os documentos que nosso RAG vai consultar.

**Em um projeto real**, esses documentos viriam de:
- PDFs de manuais
- Páginas de documentação
- Banco de dados da empresa
- Artigos científicos
- Etc.

Para esse exemplo didático, vamos usar uma lista de frases sobre **exploração de Marte**.

In [ ]:
# Base de conhecimento: cada string é um 'documento' que pode ser recuperado
documentos = [
    'A exploração de Marte revelou a presença de gelo de água nos polos do planeta.',
    'A NASA planeja enviar humanos para Marte na década de 2030, no programa Artemis.',
    'O rover Perseverance está coletando amostras de solo na cratera Jezero desde 2021.',
    'A atmosfera de Marte é composta principalmente por dióxido de carbono (cerca de 95%).',
    'Marte tem dois satélites naturais chamados Phobos e Deimos.',
    'A SpaceX, empresa de Elon Musk, também tem planos de colonizar Marte com a nave Starship.',
    'Um dia em Marte (chamado sol) dura cerca de 24 horas e 37 minutos.',
    'A temperatura média na superfície de Marte é de aproximadamente -63°C.',
    'O Olympus Mons em Marte é o maior vulcão do Sistema Solar, com 22 km de altura.',
    'A gravidade em Marte é cerca de 38% da gravidade da Terra.'
]

print(f'📚 Total de documentos na base: {len(documentos)}')
print(f'\n📄 Exemplo do primeiro documento:')
print(f'   "{documentos[0]}"')

## 4️⃣ Criação dos Embeddings

**O que são embeddings?**

Embeddings transformam texto em vetores numéricos (listas de números) que capturam o significado semântico. Frases parecidas geram vetores parecidos.

Exemplo:
- `'gato'` → `[0.21, -0.45, 0.78, ...]` (384 números)
- `'felino'` → `[0.20, -0.43, 0.79, ...]` (vetor parecido!)
- `'avião'` → `[-0.65, 0.12, -0.33, ...]` (vetor bem diferente)

Vamos usar o modelo **`all-MiniLM-L6-v2`** da HuggingFace: rápido, leve (90MB) e gratuito.

In [ ]:
# Carregar o modelo de embeddings da HuggingFace
# Na primeira execução, ele baixa o modelo (~90MB) - pode demorar 1-2 min
print('⏳ Carregando modelo de embeddings...')

modelo_embeddings = HuggingFaceEmbeddings(
    model_name='sentence-transformers/all-MiniLM-L6-v2'
)

print('✅ Modelo de embeddings carregado!')

# Vamos testar gerando um embedding
exemplo_vetor = modelo_embeddings.embed_query('Olá, mundo!')
print(f'\n🔢 Tamanho do vetor de embedding: {len(exemplo_vetor)} dimensões')
print(f'🔢 Primeiros 5 valores: {exemplo_vetor[:5]}')

## 5️⃣ Criação do Vector Database (FAISS)

**O que é FAISS?**

FAISS (Facebook AI Similarity Search) é um banco de dados vetorial otimizado para busca por similaridade. Ele armazena os embeddings dos documentos e, dada uma pergunta, encontra os mais parecidos rapidamente.

**Como funciona a busca?**

1. A pergunta é transformada em vetor (embedding)
2. FAISS calcula a distância entre esse vetor e todos os vetores dos documentos
3. Retorna os K documentos com menor distância (mais similares)

In [ ]:
# Criar o vector database a partir dos nossos documentos
# Por baixo dos panos: cada documento é convertido em vetor e indexado no FAISS
print('⏳ Criando vector database com FAISS...')

vector_db = FAISS.from_texts(
    texts=documentos,
    embedding=modelo_embeddings
)

print(f'✅ Vector DB criado com {len(documentos)} documentos indexados!')

### 🔍 Testando a busca por similaridade

Antes de plugar o LLM, vamos só ver se a parte de **retrieval** (busca) está funcionando.

In [ ]:
# Fazer uma busca de teste
pergunta_teste = 'Qual a temperatura em Marte?'

# k=3 significa 'retorna os 3 documentos mais relevantes'
documentos_relevantes = vector_db.similarity_search(pergunta_teste, k=3)

print(f'❓ Pergunta: {pergunta_teste}\n')
print('📋 Top 3 documentos mais relevantes:')
for i, doc in enumerate(documentos_relevantes, 1):
    print(f'   {i}. {doc.page_content}')

## 6️⃣ Carregando o LLM (Generator)

Agora carregamos o modelo que vai **gerar a resposta**.

**Por que FLAN-T5?**
- ✅ É **gratuito** e roda **localmente** no Colab (sem precisar de API token)
- ✅ É **leve** (~250MB), funciona até em CPU
- ✅ Foi treinado especificamente para tarefas de Q&A
- ✅ Modelo open-source da Google, hospedado na HuggingFace

**Diferença para a Inference API:** ao rodar local, evitamos problemas de rate limit, modelos depreciados ou tokens expostos.

In [ ]:
# Carregar o modelo FLAN-T5 da HuggingFace
# Como a versão nova do transformers removeu várias tasks do pipeline(),
# vamos chamar o modelo diretamente (mais robusto e didático)
print('⏳ Carregando o LLM (FLAN-T5)...')

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from langchain_core.language_models.llms import LLM
from typing import Any, List, Optional

# Carrega tokenizer e modelo
model_id = 'google/flan-t5-base'
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForSeq2SeqLM.from_pretrained(model_id)

# Criamos uma classe LLM customizada do LangChain
# Ela apenas chama model.generate() diretamente
class FlanT5LLM(LLM):
    @property
    def _llm_type(self) -> str:
        return 'flan-t5'

    def _call(self, prompt: str, stop: Optional[List[str]] = None,
              run_manager: Any = None, **kwargs: Any) -> str:
        # Tokeniza o prompt (transforma texto em IDs)
        inputs = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=512)

        # Gera a resposta
        output_ids = model.generate(
            **inputs,
            max_new_tokens=200,
            temperature=0.3,
            do_sample=True,
            top_p=0.9
        )

        # Decodifica os IDs de volta para texto
        resposta = tokenizer.decode(output_ids[0], skip_special_tokens=True)
        return resposta

# Instancia o LLM
llm = FlanT5LLM()

print('✅ LLM carregado e pronto para uso!')

## 7️⃣ Definição do Prompt Template

O **prompt** é a instrução que damos ao LLM. Em RAG, o prompt contém:
- O **contexto** recuperado da base de conhecimento
- A **pergunta** do usuário
- Uma **instrução** de como responder

Bons prompts melhoram muito a qualidade das respostas.

In [ ]:
# Criar um template reutilizável
# As variáveis {contexto} e {pergunta} serão substituídas a cada chamada
prompt_template = PromptTemplate(
    input_variables=['contexto', 'pergunta'],
    template=(
        'Use as informações do contexto abaixo para responder a pergunta de forma clara e objetiva.\n'
        'Se a resposta não estiver no contexto, diga que não sabe.\n\n'
        'Contexto: {contexto}\n\n'
        'Pergunta: {pergunta}\n\n'
        'Resposta:'
    )
)

print('✅ Prompt template definido!')
print('\n📝 Exemplo de prompt formatado:\n')
print(prompt_template.format(
    contexto='Marte é o quarto planeta do Sistema Solar.',
    pergunta='Qual a posição de Marte no sistema solar?'
))

## 8️⃣ Função RAG completa

Agora juntamos tudo em uma única função que:
1. Recebe uma pergunta
2. Busca documentos relevantes no vector DB
3. Monta o prompt com o contexto
4. Pede para o LLM gerar a resposta
5. Retorna a resposta + os documentos usados (transparência!)

In [ ]:
def responder_com_rag(pergunta: str, k: int = 3) -> dict:
    '''
    Pipeline RAG completo: busca documentos relevantes e gera uma resposta.

    Args:
        pergunta (str): pergunta do usuário
        k (int): número de documentos a recuperar (padrão: 3)

    Returns:
        dict: contendo a resposta e os documentos utilizados
    '''
    # ETAPA 1: RETRIEVAL - buscar documentos similares
    docs_relevantes = vector_db.similarity_search(pergunta, k=k)

    # Concatena os documentos em uma única string de contexto
    contexto = ' '.join([doc.page_content for doc in docs_relevantes])

    # ETAPA 2: AUGMENTATION - montar o prompt com contexto + pergunta
    prompt_final = prompt_template.format(
        contexto=contexto,
        pergunta=pergunta
    )

    # ETAPA 3: GENERATION - LLM gera a resposta
    resposta = llm.invoke(prompt_final)

    return {
        'pergunta': pergunta,
        'resposta': resposta.strip(),
        'documentos_usados': [doc.page_content for doc in docs_relevantes]
    }

print('✅ Função RAG pronta!')

## 9️⃣ Testando o RAG! 🚀

Vamos fazer várias perguntas e ver como o sistema responde.

In [ ]:
# Função auxiliar para imprimir o resultado de forma organizada
def imprimir_resultado(resultado: dict):
    print('=' * 70)
    print(f'❓ PERGUNTA: {resultado["pergunta"]}')
    print('-' * 70)
    print(f'💡 RESPOSTA: {resultado["resposta"]}')
    print('-' * 70)
    print('📚 DOCUMENTOS USADOS:')
    for i, doc in enumerate(resultado['documentos_usados'], 1):
        print(f'   {i}. {doc}')
    print('=' * 70)
    print()

In [ ]:
# Teste 1: pergunta sobre dados específicos
resultado1 = responder_com_rag('Quais são os planos da NASA para Marte?')
imprimir_resultado(resultado1)

In [ ]:
# Teste 2: pergunta sobre características físicas
resultado2 = responder_com_rag('Como é a atmosfera de Marte?')
imprimir_resultado(resultado2)

In [ ]:
# Teste 3: pergunta sobre satélites
resultado3 = responder_com_rag('Marte tem luas?')
imprimir_resultado(resultado3)

In [ ]:
# Teste 4: pergunta cuja resposta NÃO está na base
# (esperamos que o modelo diga que não sabe, em vez de inventar)
resultado4 = responder_com_rag('Qual a cor do céu em Júpiter?')
imprimir_resultado(resultado4)

## 🔟 Comparação: COM RAG vs SEM RAG

Vamos comparar a resposta do mesmo LLM **sem o contexto** (apenas com o conhecimento dele) versus **com o RAG**.

In [ ]:
pergunta = 'Quanto tempo dura um dia em Marte?'

# SEM RAG: LLM responde sem nenhum contexto
resposta_sem_rag = llm.invoke(f'Pergunta: {pergunta}\nResposta:')

# COM RAG: usa nossa função
resultado_com_rag = responder_com_rag(pergunta)

print('=' * 70)
print(f'❓ PERGUNTA: {pergunta}')
print('=' * 70)
print(f'\n❌ SEM RAG: {resposta_sem_rag.strip()}')
print(f'\n✅ COM RAG: {resultado_com_rag["resposta"]}')
print('=' * 70)

## 📊 Conclusão

Implementamos com sucesso um sistema RAG completo usando:

✅ **Embeddings** com `sentence-transformers` da HuggingFace  
✅ **Vector Database** com FAISS (busca por similaridade)  
✅ **LLM** com o modelo `flan-t5-base` da HuggingFace, rodando localmente  
✅ **Orquestração** com LangChain  

### Vantagens do RAG

- 🎯 **Respostas baseadas em fatos**: o LLM responde com base nos documentos fornecidos
- 🔄 **Conhecimento atualizável**: basta adicionar novos documentos à base
- 🔍 **Transparência**: sabemos quais documentos foram usados na resposta
- 💰 **Custo reduzido**: não precisa retreinar o LLM com novos dados

### Próximos passos

Para tornar essa aplicação ainda mais robusta, poderíamos:
- Carregar documentos reais de PDFs (`PyPDFLoader` do LangChain)
- Dividir documentos longos em pedaços menores (`RecursiveCharacterTextSplitter`)
- Usar um LLM maior (Llama 3, Mistral) com mais qualidade
- Adicionar uma interface web (Streamlit ou Gradio)
- Implementar avaliação de qualidade das respostas (RAGAS)

---

## 📖 Referências

- [HuggingFace](https://huggingface.co/)
- [LangChain Docs](https://python.langchain.com/docs/get_started/introduction)
- [FAISS](https://github.com/facebookresearch/faiss)
- [Sentence Transformers](https://www.sbert.net/)
- [FLAN-T5 Paper (Google)](https://arxiv.org/abs/2210.11416)
- [RAG Original Paper (Lewis et al., 2020)](https://arxiv.org/abs/2005.11401)